<a href="https://colab.research.google.com/github/anilpomar/Full-Stack-Gen-AI-BootCamp-KrishNaik-/blob/main/Assignment_non_instruction_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install stable torch (2.5.1 is more stable than 2.4.0)
!pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.51.0
!pip install trl==0.23.0
!pip install peft accelerate bitsandbytes xformers

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 92.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 118.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1

In [ ]:
# Install unsloth from PyPI (NOT git) - more stable
!pip install -q unsloth transformers==5.5.0 trl==0.24.0 peft accelerate bitsandbytes
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 802.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [45]:
# -------------------------
# 2. Imports
# -------------------------

import torch
import unsloth

print(f"Torch: {torch.__version__}")  # Should be 2.5.1
print(f"Unsloth: {unsloth.__version__}")

import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset


from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

Torch: 2.10.0+cu128
Unsloth: 2026.6.9
DPO patch applied.
GPU: Tesla T4


In [56]:
from dataclasses import dataclass
# -------------------------
# 3. File paths
# -------------------------

non_instruction_data_path = "/content/non_instruction_dataset.pdf"
instruction_data_path = "/content/instruction_dataset.jsonl"
preference_data_path = "/content/preference_dataset.jsonl"

for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
  if not os.path.exists(path):
      raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
# 4. Configuration
# -------------------------
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH =2048 #512
SEED =3407 # was 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 2
GRAD_ACCUM_STEPS =8  #was 2 #16
WARMUP_STEPS = 5 #10
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 15
STAGE2_MAX_STEPS = 20
STAGE3_MAX_STEPS = 20

STAGE1_LR = 5e-5
STAGE1_EMBR=5e-5
STAGE2_LR = 1e-4
STAGE3_LR = 5e-5

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"

non_instruction_ADAPTER_DIR = f"{OUTPUT_ROOT}/non_instruction_adapter"
non_instruction_MERGED_DIR  = f"{OUTPUT_ROOT}/non_instruction_merged_model"

instruction_ADAPTER_DIR = f"{OUTPUT_ROOT}/instruction_adapter"
instruction_MERGED_DIR  = f"{OUTPUT_ROOT}/instruction_merged_model"

preference_ADAPTER_DIR = f"{OUTPUT_ROOT}/preference_dpo_adapter"
preference_FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/preference_dpo_final_merged_model"

for path in [
  OUTPUT_ROOT,
  non_instruction_ADAPTER_DIR,
  non_instruction_MERGED_DIR,
  instruction_ADAPTER_DIR,
  instruction_MERGED_DIR,
  preference_ADAPTER_DIR,
  preference_FINAL_MERGED_DIR,
]:
  os.makedirs(path, exist_ok=True)


In [57]:
# ============================================================
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================

def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages

def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)

In [58]:
# -------------------------
# 5. Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj","embed_tokens", "lm_head"
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer

def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train() #Model Training go here.
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result

def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [59]:
non_instruction_dataset = build_pdf_dataset(non_instruction_data_path)

PDF pages extracted: 26
Paragraph records: 25

Sample paragraph:
 Exhibit 10.3 Manufacturing Agreement Between Antares Pharma, Inc. and AMAG Pharmaceuticals, Inc. MANUFACTURING AGREEMENT This Manufacturing Agreement ("Agreement") is made and entered into as of the 20th day of March, 2018 (the "Effective Date") by and between Antares Pharma, Inc., a Delaware corporation, with offices located at 100 Princeton South, Suite 300, Ewing, NJ 08628 ("Antares"), and AMAG Pharmaceuticals, Inc., a Delaware corporation, with a corporate address at 1100 Winter Street, Waltham, MA 02451 ("AMAG"). Antares and AMAG are sometimes referred to herein individually as a "Party" and collectively as the "Parties". Recitals WHEREAS, AMAG is engaged in discovering, developing and 


In [60]:

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    weight_decay = 0.01,
    max_grad_norm = 0.3,
    lr_scheduler_type = "cosine",
    seed=SEED,
    warmup_ratio = 0.1,
    # embedding_learning_rate=  STAGE1_EMBR
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=non_instruction_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION  TRAINING")


save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=non_instruction_ADAPTER_DIR,
    merged_dir=non_instruction_MERGED_DIR,
    stage_name="Stage 1",
)

# #Plot Loss Graph
# losses = [h["loss"] for h in stage1_trainer.state.log_history if "loss" in h]
# losses = np.array(losses)
# k = 5
# roll = np.convolve(losses, np.ones(k)/k, mode="valid")
# roll_x = np.arange(k//2, k//2 + len(roll))   # center the window

# plt.figure(figsize=(9, 4))
# plt.plot(losses, ":", color="#898781", marker=".", label="raw loss")
# plt.plot(roll_x, roll, "-", color="#2a78d6", linewidth=2.5, label="rolling avg (5)")
# plt.axhline(losses.mean(), color="#b4b2a9", linestyle="--", label=f"mean {losses.mean():.3f}")
# plt.xlabel("step"); plt.ylabel("training loss"); plt.legend()
# plt.title(f"range={losses.max()-losses.min():.3f}  drift={roll[-1]-roll[0]:+.3f}")
# plt.show()

del stage1_trainer
del stage1_model
clear_gpu_memory()


==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


Unsloth: Training embed_tokens in mixed precision to save VRAM


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...


STAGE 1 - NON-INSTRUCTION  TRAINING RESULTS
Train time/sec: 79.67
Peak allocated VRAM/GB: 4.009
Peak reserved VRAM/GB: 4.273

Saving Stage 1 adapter...
Stage 1 adapter saved to: /content/unsloth_pharma_merge_reload_outputs/non_instruction_adapter

Merging Stage 1 adapter with base model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model`: 100%|██████████| 1/1 [01:01<00:00, 61.14s/it]


Successfully copied all 1 files from cache to `/content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:54<00:00, 54.22s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model
